# <div align="center">Odlotowe projekty</div>
---

<img src="https://sprawdzai.org/static/files/image/661ffee8-bde3-453e-a4a0-01c941467b2d" width="700" />

Zespół inżynierów przygotowuje nową rakietę do misji badawczej. Każdy projekt opisany jest przez 12 parametrów konstrukcyjnych, takich jak własności silników, zbiorników paliwa, aerodynamiki i rozkładu masy.

Dokładne zasymulowanie lotu każdego projektu jest kosztowne obliczeniowo, dlatego chcemy zbudować model, który na podstawie parametrów konstrukcji szybko przewidzi cztery najważniejsze wyniki symulacji: maksymalną osiągniętą wysokość, maksymalną prędkość, zużycie paliwa oraz maksymalne przeciążenie.

Twoim zadaniem jest zbudowanie wielowarstwowego perceptronu (MLP), który będzie w stanie przybliżać wyniki symulatora.

## Pobranie danych

Otrzymujesz zbiór treningowy i walidacyjny. Zbiór walidacyjny używany na SprawdzAI jest dokładnie tym samym zbiorem, który pobierzesz w tym notatniku. Ukryty zbiór testowy zawiera nowe dane służące do finalnej oceny Twojego rozwiązania. Dane testowe mają ten sam format i pochadzą z tej samej dystrybucji co zbiór treningowy oraz walidacyjny.

- Zbiór treningowy: `features` o kształcie `(4096, 12)` i `targets` o kształcie `(4096, 4)`
- Zbiór walidacyjny i testowy: `features` o kształcie `(1536, 12)` i `targets` o kształcie `(1536, 4)`
- Typ wszystkich tensorów: `torch.float32`

Musisz stworzyć i wytrenować model, który otrzymując na wejściu `features` zwróci jak najlepiej przybliżone wartości `targets`.

In [1]:
from pathlib import Path
from zipfile import ZipFile

FINAL_EVALUATION_MODE = False  # Na SprawdzAI flaga ta zostanie automatycznie zmieniona na True

if not Path('train.pt').exists() or not Path('valid.pt').exists():
    !curl -o dane.zip https://sprawdzai.org/static/files/dataset/cbd52f0a-8c11-4ddc-850d-7901b4ab01c9
    with ZipFile('dane.zip') as archive:
        archive.extractall()

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  328k  100  328k    0     0   240k      0  0:00:01  0:00:01 --:--:--  240k


In [2]:
import torch

train = torch.load('train.pt', map_location='cpu', weights_only=True)

train_features = train['features']
train_targets = train['targets']

print('Zbiór treningowy:', train_features.shape, train_targets.shape)

Zbiór treningowy: torch.Size([4096, 12]) torch.Size([4096, 4])


## Zbuduj MLP

Zastąp poniższy szkielet modelu poprawną implementacją sieci neuronowej:
- Użyj wybranej liczby warstw `nn.Linear`, a pomiędzy nimi - funkcje aktywacji, na przykład `nn.ReLU()` czy `nn.SiLU()`.
- Pierwsza warstwa powinna przyjmować 12 wartości i zwracać wybraną przez Ciebie ilość ukrytych neuronów.
- Ostatnia warstwa powinna zwracać 4 wartości.
- Pamiętaj, że przed pierwszą warstwą oraz po ostatniej warstwie nie powinny znajdować się funkcje aktywacji - należy je dodawać wyłącznie **pomiędzy** wszystkimi warstwami liniowymi.

Jeśli chcesz, możesz dać dodatkowe argumenty dla funkcji `__init__`. Nie zmieniaj natomiast argumentów funkcji `forward` - to ona zostanie później uruchomiona na zbiorze walidacyjnym lub testowym, aby ocenić Twoje rozwiązanie.

In [3]:
from torch import nn

class activate(nn.Module):
  def __init__(self, in_features, out_features):
    super().__init__()
    self.linear = nn.Linear(in_features, out_features)
  def forward(self, x):
    x = self.linear(x)
    x = nn.functional.relu(x)
    return x

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # Przystosowujemy wymiary: wejście to 12 cech, wyjście to 4 wyniki
        self.lay1 = activate(12, 64)
        self.lay2 = activate(64, 32)
        self.lay3 = activate(32, 16)
        self.lay4 = nn.Linear(16, 4) # Ostatnia warstwa liniowa bez aktywacji

    def forward(self, features):
        x = self.lay1(features)
        x = self.lay2(x)
        x = self.lay3(x)
        x = self.lay4(x)
        return x

## Naucz model

Trenowanie modelu polega na wielokrotnym zaktualizowaniu parametrów korzystając z aktualnych gradientów błędu modelu. Napisz pętlę treningową dla swojego modelu. W każdej iteracji wykonaj pięć kroków:

1. Oblicz predykcje modelu dla `train_features`.
2. Korzystając z tych predykcji i oczekiwanych odpowiedzi `train_targets`, policz błąd modelu. Użyj błędu średniokwadratowego (*Mean Squared Error - MSE*).
3. Wywołaj `loss.backward()` aby obliczyć gradienty. Dzięki temu w każdym parametrze modelu pojawi się atrubyt `.grad` zawierający jego gradienty.
4. Zaktualizuj parametry modelu - od każdego parametru odejmij jego gradient pomnożony przez `learning_rate`.
5. Wyzeruj gradienty przed kolejną iteracją.

Jeśli korzystasz z wbudowanego optymalizatora PyTorch, kroki 4 i 5 wykonasz uruchamiając funkcje `optimizer.step()` i `optimizer.zero_grad()`.

Twoje pierwsze podejście może nie zdobyć od razu zdobyć 100 punktów. Spróbuj pobawić się nieco architekturą i hiperparametrami treningu, aby zdobyć wyższy wynik. Za duży model może przetrenować się do danych treningowych, a za mały może nie być w stanie dobrze rozwiązać problemu - gdy już napiszesz pętlę treningową i przetestujesz wstępnie swoje rozwiązanie, spróbuj różnych konfiguracji liczby warstw, liczby neuronów, `learning_rate` i `steps` aby uzyskać wyższy wynik.

Możesz również spróbować różne funkcje aktywacji oraz różne liczby neuronów na kolejnych warstwach, np. zamiast ustawiać stałą ilość ukrytych neuronów dla każdej warstwy (np. `12 -> 32 -> 32 -> 32 -> 4`), popularnym sposobem jest dzielenie ilości neuronów przez 2 po każdej warsrwie (np. `12 -> 64 -> 32 -> 16 -> 4`).

Poniższa komórka przenosi parametry modelu oraz dane zbioru treningowego na GPU, aby przyspieszyć obliczenia.

In [4]:
import torch.optim as optim

# Dynamiczny wybór urządzenia (GPU jeśli dostępne, w przeciwnym razie CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MLP().to(device)
train_features_dev = train_features.to(device)
train_targets_dev = train_targets.to(device)

learning_rate = 0.005
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
steps = 2000

losses = []

model.train()
for step in range(steps):
    # 1. Obliczenie predykcji
    predictions = model(train_features_dev)

    # 2. Obliczenie błędu średniokwadratowego (MSE)
    loss = ((predictions - train_targets_dev) ** 2).mean()

    # 3. Wsteczna propagacja błędu
    optimizer.zero_grad()
    loss.backward()

    # 4. Aktualizacja parametrów
    optimizer.step()

    # Zapisujemy historię błędu
    losses.append(loss.item())

    # Wypisujemy status tekstowo co 200 kroków
    if (step + 1) % 200 == 0 or step == steps - 1:
        print(f'Krok {step + 1}/{steps} - Aktualny błąd: {loss.item():.5f}')

Krok 200/2000 - Aktualny błąd: 0.22238
Krok 400/2000 - Aktualny błąd: 0.08112
Krok 600/2000 - Aktualny błąd: 0.06707
Krok 800/2000 - Aktualny błąd: 0.05320
Krok 1000/2000 - Aktualny błąd: 0.04886
Krok 1200/2000 - Aktualny błąd: 0.04597
Krok 1400/2000 - Aktualny błąd: 0.04187
Krok 1600/2000 - Aktualny błąd: 0.04019
Krok 1800/2000 - Aktualny błąd: 0.03849
Krok 2000/2000 - Aktualny błąd: 0.03724


## Ewaluacja rozwiązania

Metryką jest **MSE**, czyli średni błąd kwadratowy liczony po wszystkich przykładach i ich czterech wyjściach. Im mniejsza jest ta wartość, tym lepiej. Ta komórka korzysta z publicznego zbioru walidacyjnego i nie uruchamia się podczas oceny na SprawdzAI.

Aby Twoje rozwiązanie zostało poprawnie odczytane przez SprawdzAI, nie zmieniaj nazwy zmiennej `model`. SprawdzAI przekaże modelowi tensor `torch.float32` o kształcie `(N, 12)` i oczekuje tensora o kształcie `(N, 4)` z wartościami skończonymi.

🔴 MSE równe **1.0** lub większe daje 0 punktów. To wynik predykcji średnią ze zbioru treningowego.

🟢 MSE równe **0.1** lub mniejsze daje 100 punktów.

🟡 Pomiędzy tymi progami wynik jest przeliczany liniowo.

In [5]:
def evaluate():
    valid = torch.load('valid.pt', map_location='cpu', weights_only=True)

    # Używamy tego samego dynamicznego wyboru urządzenia
    eval_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    features = valid['features'].to(eval_device)
    targets = valid['targets'].to(eval_device)

    model.to(eval_device)
    model.eval()
    with torch.no_grad():
        predictions = model(features)

    mse = ((predictions - targets) ** 2).mean().item()

    return mse

In [6]:
if not FINAL_EVALUATION_MODE:
    validation_mse = evaluate()
    print(f'MSE na zbiorze walidacyjnym: {validation_mse:.5f}')

MSE na zbiorze walidacyjnym: 0.07884
